# PostgreSQL 연결 테스트

PostgreSQL 데이터베이스 연결 및 기본 쿼리 테스트

In [ ]:
import sys
sys.path.insert(0, '../..')

import psycopg2
from config.settings import POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB, POSTGRES_USER, POSTGRES_PASSWORD

print("📌 PostgreSQL 연결 정보:")
print(f"  Host: {POSTGRES_HOST}")
print(f"  Port: {POSTGRES_PORT}")
print(f"  Database: {POSTGRES_DB}")
print(f"  User: {POSTGRES_USER}")

## 1. 연결 테스트

In [ ]:
try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    print("✅ PostgreSQL 연결 성공!")
    conn.close()
except Exception as e:
    print(f"❌ 연결 실패: {e}")

## 2. 테이블 목록 확인

In [ ]:
try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    cursor = conn.cursor()
    
    # 테이블 목록 조회
    cursor.execute("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
    """)
    
    tables = cursor.fetchall()
    print(f"📊 총 {len(tables)}개의 테이블:")
    for table in tables:
        print(f"  - {table[0]}")
    
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ 테이블 조회 실패: {e}")

## 3. 각 테이블 레코드 개수 확인

In [ ]:
import pandas as pd

try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    cursor = conn.cursor()
    
    tables = [
        'raw_clickstream_events',
        'clickstream_events',
        'daily_statistics',
        'data_quality_results',
        'error_logs',
        'error_statistics'
    ]
    
    print("📈 테이블별 레코드 개수:")
    print()
    
    for table in tables:
        try:
            cursor.execute(f"SELECT COUNT(*) FROM {table}")
            count = cursor.fetchone()[0]
            print(f"  {table:<30} : {count:>10,}개")
        except:
            print(f"  {table:<30} : (테이블 없음)")
    
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ 조회 실패: {e}")

## 4. raw_clickstream_events 샘플 데이터 조회

In [ ]:
try:
    conn = psycopg2.connect(
        host=POSTGRES_HOST,
        port=POSTGRES_PORT,
        database=POSTGRES_DB,
        user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )
    
    # Pandas로 조회하면 더 보기 좋음
    df = pd.read_sql_query(
        "SELECT * FROM raw_clickstream_events LIMIT 5",
        conn
    )
    
    print("✅ raw_clickstream_events 샘플 데이터 (5개):")
    print()
    print(df.to_string())
    
    conn.close()
except Exception as e:
    print(f"❌ 조회 실패: {e}")

## 5. Connection Pool 테스트

In [ ]:
from config.database import get_db_connection, return_db_connection, test_pool_connection

print("🔄 Connection Pool 테스트:")
print()

try:
    result = test_pool_connection()
    if result:
        print("✅ Pool 연결 테스트 성공!")
    else:
        print("❌ Pool 연결 테스트 실패!")
except Exception as e:
    print(f"❌ 테스트 실패: {e}")